# Ordinary Kriging — Final Holdout Evaluation

**Inputs (read from S3):**
- `s3://thesis-data-ismaktam/kriging/winner.json` — best (transform × variogram) from `kriging_eval.ipynb`
- `s3://thesis-data-ismaktam/kriging/global_variogram.pkl` — climatological variogram
- `s3://thesis-data-ismaktam/kriging/holdout_station_ids.json` — 492 held-out station IDs

**Outputs (written to S3):**
- `s3://thesis-data-ismaktam/kriging/test/holdout_predictions.parquet`
- `s3://thesis-data-ismaktam/kriging/test/test_metrics.json`

**What this notebook does:**
1. Loads the winning (transform × variogram) configuration from `winner.json` (S3).
2. Splits stations into 1966 train + 492 holdout (via `holdout_station_ids.json`).
3. Trains transform pipeline + monthly TPS normals on the **1966 train stations only** (no fold splits — uses full training data for max statistical power on the final eval).
4. Reuses the global climatological variogram from `kriging_train.ipynb` (Haylock 2008: same variogram for train and final eval).
5. Predicts at the 492 holdout coords and uploads `test/holdout_predictions.parquet` to S3.
6. Reports final test metrics (CRPS / MAE / RMSE) and compares to k-fold means.

## 0. Imports + paths

In [1]:
import sys, os, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import boto3
import properscoring as ps
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')

_NB = Path.cwd()
while _NB != _NB.parent and not (_NB / 'pyproject.toml').exists():
    _NB = _NB.parent
if str(_NB / 'src') not in sys.path:
    sys.path.insert(0, str(_NB / 'src'))
os.chdir(_NB)

# ---- Local cache (mirror of S3) ----
OUT_DIR  = Path('outputs/kriging')
TEST_DIR = OUT_DIR / 'test'
for d in (OUT_DIR, TEST_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- S3 ----
S3_BUCKET = 'thesis-data-ismaktam'
S3_ROOT   = 'kriging'
FORCE_RECOMPUTE = False

RANDOM_SEED  = 42
K_MC         = 100
N_JOBS       = int(os.environ.get('N_JOBS', '-1'))
N_TEST_DAYS  = int(os.environ['N_TEST_DAYS']) if os.environ.get('N_TEST_DAYS') else None

print(f'Project root: {_NB}')
print(f'S3:           s3://{S3_BUCKET}/{S3_ROOT}/')

/app/.venv/lib/python3.12/site-packages/properscoring/_crps.py:257: SyntaxWarning: invalid escape sequence '\i'
  CRPS(F, x) = \int_z (F(z) - H(z - x))^2 dz


Project root: /root/precip_interpolation_thesis
S3:           s3://thesis-data-ismaktam/kriging/


/app/.venv/lib/python3.12/site-packages/properscoring/_brier.py:10: SyntaxWarning: invalid escape sequence '\i'
  The Brier score (BS) scores binary forecasts $k \in \{0, 1\}$,
/app/.venv/lib/python3.12/site-packages/properscoring/_brier.py:102: SyntaxWarning: invalid escape sequence '\i'
  CRPS(F, x) = \int_z BS(F(z), H(z - x)) dz


## 1. S3 helpers

In [2]:
s3 = boto3.client('s3')

def s3_exists(s3_key: str) -> bool:
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=s3_key)
        return True
    except Exception:
        return False

def s3_upload(local_path: Path, s3_key: str) -> None:
    try:
        s3.upload_file(str(local_path), S3_BUCKET, s3_key)
        print(f'  ↑ s3://{S3_BUCKET}/{s3_key}')
    except Exception as e:
        print(f'  S3 upload failed: {e.__class__.__name__}: {e}')

def s3_download(s3_key: str, local_path: Path) -> bool:
    try:
        local_path.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(S3_BUCKET, s3_key, str(local_path))
        print(f'  ↓ s3://{S3_BUCKET}/{s3_key}')
        return True
    except Exception:
        return False

def fetch(s3_key: str, local_path: Path) -> bool:
    """local cache → S3 → False."""
    if local_path.exists() and not FORCE_RECOMPUTE:
        return True
    if FORCE_RECOMPUTE:
        return False
    return s3_download(s3_key, local_path)

## 2. Load winner + variogram + holdout IDs (all from S3)

In [3]:
WINNER_LOCAL  = OUT_DIR / 'winner.json'
VGM_LOCAL     = OUT_DIR / 'global_variogram.pkl'
HOLDOUT_LOCAL = OUT_DIR / 'holdout_station_ids.json'

if not fetch(f'{S3_ROOT}/winner.json', WINNER_LOCAL):
    raise FileNotFoundError(f's3://{S3_BUCKET}/{S3_ROOT}/winner.json missing — run kriging_eval.ipynb first')
if not fetch(f'{S3_ROOT}/global_variogram.pkl', VGM_LOCAL):
    raise FileNotFoundError(f's3://{S3_BUCKET}/{S3_ROOT}/global_variogram.pkl missing — run kriging_train.ipynb first')

# holdout_station_ids.json: try S3 first, then fall back to project-local outputs/holdout_station_ids.json
if not fetch(f'{S3_ROOT}/holdout_station_ids.json', HOLDOUT_LOCAL):
    legacy = Path('outputs/holdout_station_ids.json')
    if legacy.exists():
        print(f'  Using local fallback: {legacy} — uploading to S3 for future runs')
        s3_upload(legacy, f'{S3_ROOT}/holdout_station_ids.json')
        HOLDOUT_LOCAL = legacy
    else:
        raise FileNotFoundError(
            f'holdout_station_ids.json not found in S3 ({S3_ROOT}/holdout_station_ids.json) or locally ({legacy})'
        )

with open(WINNER_LOCAL) as f:
    winner = json.load(f)
with open(VGM_LOCAL, 'rb') as f:
    payload = pickle.load(f)
# Pickle from kriging_train is a wrapped payload {'variograms': {(t,vm): info}, 'empirical': ..., 'lag_centers_km': ...}
# Unwrap to the inner {(t, vm) -> info} dict the worker expects.
global_vgm = payload['variograms'] if isinstance(payload, dict) and 'variograms' in payload else payload
assert all(isinstance(k, tuple) and len(k) == 2 for k in global_vgm), \
    f'global_vgm has unexpected keys: {list(global_vgm)[:5]} — expected tuples of (transform, variogram_model)'
holdout_ids = set(json.loads(HOLDOUT_LOCAL.read_text()))

print('Winner config:')
print(json.dumps(winner, indent=2))
print(f'\n#holdout stations: {len(holdout_ids)}')
TRANSFORMS = [winner['transform']]
VGM_MODELS = [winner['variogram_model']]

  ↓ s3://thesis-data-ismaktam/kriging/winner.json
  ↓ s3://thesis-data-ismaktam/kriging/global_variogram.pkl
  ↓ s3://thesis-data-ismaktam/kriging/holdout_station_ids.json
Winner config:
{
  "transform": "none",
  "variogram_model": "exponential",
  "mean_crps": 0.4079,
  "mean_mae": 0.5131,
  "mean_rmse": 1.3745
}

#holdout stations: 492


## 3. Load all stations (train + holdout) and fit pipeline on train only

In [4]:
from thesis.config import Config
from thesis.data.registry import DataRegistry
from thesis.transforms import (
    ProjectionTransform, IndicatorTransform,
    DetrendTransform, NormalScoreTransform, LogTransform, KrigingTransform,
)
from thesis.transforms.kriging_transform import TRANSFORMS as KT_NAMES
from thesis.transforms.pipeline import TransformPipeline
from thesis.scripts._common import FwdFn, InvFn

cfg = Config()
registry = DataRegistry.from_config(cfg)

all_raw = registry.stations.load(cfg.date_start, cfg.date_end, exclude_holdout=False)
is_holdout = all_raw['station_id'].isin(holdout_ids)
raw_train = all_raw[~is_holdout].copy()
raw_test  = all_raw[is_holdout].copy()
print(f'Train rows: {len(raw_train):,} ({raw_train["station_id"].nunique()} stations)')
print(f'Test rows : {len(raw_test):,} ({raw_test["station_id"].nunique()} stations)')

# Fit pipeline on train only — projection + indicator + detrend
proj = ProjectionTransform(target_crs=cfg.study_area.target_crs)
ind  = IndicatorTransform(threshold_mm=cfg.wet_day_threshold_mm)
det  = DetrendTransform()
base_pipeline = TransformPipeline([proj, ind, det])
proc_train = base_pipeline.fit_transform(raw_train)
proc_test  = base_pipeline.apply(raw_test)

# NST CDF fit on train wet quotas only
ns = NormalScoreTransform()
ns.fit(proc_train)
log_t = LogTransform(offset=cfg.log_transform_offset)
kts = {k: KrigingTransform(kind=k, ns=ns, log=log_t) for k in KT_NAMES}
fwd = FwdFn(kts)
inv = InvFn(kts)

proc_train_by_date = {str(d): grp for d, grp in proc_train.groupby('date')}
proc_test_by_date  = {str(d): grp for d, grp in proc_test.groupby('date')}
all_dates = sorted(set(proc_train_by_date) & set(proc_test_by_date))
if N_TEST_DAYS is not None:
    rng = np.random.default_rng(RANDOM_SEED)
    all_dates = sorted(rng.choice(all_dates, size=min(N_TEST_DAYS, len(all_dates)), replace=False).tolist())
print(f'Days to evaluate: {len(all_dates):,}')

Train rows: 45,237,660 (1966 stations)
Test rows : 11,320,920 (492 stations)
Days to evaluate: 23,010


## 4. Monthly TPS normals at holdout station coords (fit on train only)

In [5]:
from thesis.models.kriging.monthly_norms import build_monthly_norms_at_points

NORMS_LOCAL = TEST_DIR / 'holdout_norms.pkl'
NORMS_S3    = f'{S3_ROOT}/test/holdout_norms.pkl'

if fetch(NORMS_S3, NORMS_LOCAL):
    print(f'Cached norms: {NORMS_LOCAL}')
    with open(NORMS_LOCAL, 'rb') as f:
        payload = pickle.load(f)
    test_meta = payload['test_meta']
    norms_2d  = payload['norms_2d']
    norms_3d  = payload['norms_3d']
else:
    test_meta = (
        proc_test[['station_id', 'x_proj', 'y_proj']
                  + (['elevation_m'] if 'elevation_m' in proc_test.columns else [])]
        .drop_duplicates('station_id').reset_index(drop=True)
    )
    elev = test_meta['elevation_m'].values if 'elevation_m' in test_meta.columns else None
    norms_2d, norms_3d = build_monthly_norms_at_points(
        det, proc_train,
        test_meta['x_proj'].values, test_meta['y_proj'].values, elev,
    )
    with open(NORMS_LOCAL, 'wb') as f:
        pickle.dump({'test_meta': test_meta, 'norms_2d': norms_2d, 'norms_3d': norms_3d}, f)
    s3_upload(NORMS_LOCAL, NORMS_S3)

test_sid_to_idx = {s: i for i, s in enumerate(test_meta['station_id'].values)}
print(f'Norms 2D shape: {norms_2d.shape}   Norms 3D: {None if norms_3d is None else norms_3d.shape}')

  ↓ s3://thesis-data-ismaktam/kriging/test/holdout_norms.pkl
Cached norms: outputs/kriging/test/holdout_norms.pkl
Norms 2D shape: (12, 492)   Norms 3D: (12, 492)


## 5. Predict at holdout stations — winner combo, both 2D and 3D TPS

Two prediction passes for the same winning (transform × variogram), differing only in how the climatological monthly normal $\bar M(\mathbf{s}_0, m)$ is interpolated from the 1966 train stations to the 492 holdout coordinates:
- **2D TPS** — surface fitted on $(x, y)$ alone
- **3D TPS** — surface fitted on $(x, y, z_{\text{elev}})$, capturing orographic enhancement

Same fitted variogram, same kriging system, same Monte Carlo back-transformation; only the per-cell monthly normal differs. The 3D pass is the headline holdout result; comparing the two on the same fully held-out 492 stations gives the elevation contribution. Output column `tps_mode ∈ {2D, 3D}` distinguishes the rows.

In [6]:
from thesis.models.kriging.kfold_worker import predict_one_day_all_combos

MAX_WET        = cfg.kriging.max_wet
N_STATIONS_MIN = cfg.kriging.n_stations_min

OUT_LOCAL = TEST_DIR / 'holdout_predictions.parquet'
OUT_S3    = f'{S3_ROOT}/test/holdout_predictions.parquet'

if fetch(OUT_S3, OUT_LOCAL):
    print(f'Cached: {OUT_LOCAL} — loading')
    df_test = pd.read_parquet(OUT_LOCAL)
else:
    # Build per-day job list once — same data, both modes will reuse it.
    jobs = []
    for date in all_dates:
        proc_train_d = proc_train_by_date[date]
        proc_test_d  = proc_test_by_date[date]
        if len(proc_test_d) == 0:
            continue
        test_idx = np.array([test_sid_to_idx[s] for s in proc_test_d['station_id'].values])
        n2d_today = norms_2d[:, test_idx]
        n3d_today = norms_3d[:, test_idx] if norms_3d is not None else None
        jobs.append((date, proc_train_d, proc_test_d, n2d_today, n3d_today))
    print(f'Submitting {len(jobs):,} day-jobs to {N_JOBS} workers (per mode)...')

    modes_to_run = [('2D', False)]
    if norms_3d is not None:
        modes_to_run.append(('3D', True))

    all_dfs = []
    for tps_mode_label, use_3d in modes_to_run:
        print(f'\n→ TPS mode = {tps_mode_label}')
        results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=5)(
            delayed(predict_one_day_all_combos)(
                date, proc_train_d, proc_test_d,
                global_vgm, TRANSFORMS, VGM_MODELS,
                fwd, inv, n2d_today, n3d_today,
                N_STATIONS_MIN, MAX_WET, K_MC, RANDOM_SEED,
                use_3d,
            )
            for date, proc_train_d, proc_test_d, n2d_today, n3d_today in jobs
        )
        mode_df = pd.concat([d for d in results if not d.empty], ignore_index=True)
        mode_df['tps_mode'] = tps_mode_label
        all_dfs.append(mode_df)
        print(f'  {tps_mode_label}: {len(mode_df):,} records')

    df_test = pd.concat(all_dfs, ignore_index=True)
    df_test.to_parquet(OUT_LOCAL, index=False)
    s3_upload(OUT_LOCAL, OUT_S3)
    print(f'\nSaved {len(df_test):,} total records → {OUT_LOCAL}')

Submitting 23,010 day-jobs to -1 workers (per mode)...

→ TPS mode = 2D


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 185 concurrent workers.
[Parallel(n_jobs=-1)]: Done  80 tasks      | elapsed:   17.0s
[Parallel(n_jobs=-1)]: Done 278 tasks      | elapsed:   24.5s
[Parallel(n_jobs=-1)]: Done 512 tasks      | elapsed:   29.2s
[Parallel(n_jobs=-1)]: Done 782 tasks      | elapsed:   34.3s
[Parallel(n_jobs=-1)]: Done 1088 tasks      | elapsed:   40.3s
[Parallel(n_jobs=-1)]: Done 1430 tasks      | elapsed:   47.3s
[Parallel(n_jobs=-1)]: Done 1808 tasks      | elapsed:   54.7s
[Parallel(n_jobs=-1)]: Done 2222 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 2672 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 3158 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done 3680 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 4238 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 4832 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 5462 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 6128 tasks      

  2D: 11,320,920 records

→ TPS mode = 3D


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 185 concurrent workers.
[Parallel(n_jobs=-1)]: Done  80 tasks      | elapsed:    5.2s
[Parallel(n_jobs=-1)]: Done 278 tasks      | elapsed:    9.2s
[Parallel(n_jobs=-1)]: Done 512 tasks      | elapsed:   13.8s
[Parallel(n_jobs=-1)]: Done 782 tasks      | elapsed:   18.7s
[Parallel(n_jobs=-1)]: Done 1088 tasks      | elapsed:   26.5s
[Parallel(n_jobs=-1)]: Done 1430 tasks      | elapsed:   33.9s
[Parallel(n_jobs=-1)]: Done 1808 tasks      | elapsed:   41.5s
[Parallel(n_jobs=-1)]: Done 2222 tasks      | elapsed:   49.8s
[Parallel(n_jobs=-1)]: Done 2672 tasks      | elapsed:   58.6s
[Parallel(n_jobs=-1)]: Done 3158 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 3680 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 4238 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 4832 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 5462 tasks      | elapsed:  1.9min
[Parallel(n_jobs=-1)]: Done 6128 tasks      

  3D: 11,320,920 records
  ↑ s3://thesis-data-ismaktam/kriging/test/holdout_predictions.parquet

Saved 22,641,840 total records → outputs/kriging/test/holdout_predictions.parquet


## 6. Final test metrics → S3

In [7]:
def _metrics(grp):
    obs  = grp['observed_mm'].values
    pred = grp['predicted_mm'].values
    sd   = np.sqrt(np.maximum(grp['predicted_var_mm2'].values, 1e-8))
    return {
        'crps':       float(np.mean(ps.crps_gaussian(obs, mu=pred, sig=sd))),
        'mae':        float(np.mean(np.abs(obs - pred))),
        'rmse':       float(np.sqrt(np.mean((obs - pred) ** 2))),
        'n':          int(len(grp)),
        'n_stations': int(grp['station_id'].nunique()),
    }

PRED_WET = 0.4

# Per-mode metrics: full dataset and predicted-wet subset
metrics_per_mode = {}
for mode, grp in df_test.groupby('tps_mode'):
    pw = grp[grp['predicted_mm'] >= PRED_WET]
    metrics_per_mode[mode] = {
        'all':           _metrics(grp),
        'predicted_wet': _metrics(pw) if len(pw) else None,
    }

metrics = {
    'transform':        winner['transform'],
    'variogram_model':  winner['variogram_model'],
    'tps_modes':        sorted(metrics_per_mode.keys()),
    'per_mode':         metrics_per_mode,
    'kfold_mean_crps':  winner['mean_crps'],
    'kfold_mean_mae':   winner['mean_mae'],
    'kfold_mean_rmse':  winner['mean_rmse'],
}
metrics_local = TEST_DIR / 'test_metrics.json'
with open(metrics_local, 'w') as f:
    json.dump(metrics, f, indent=2)
s3_upload(metrics_local, f'{S3_ROOT}/test/test_metrics.json')

print('=== FINAL HOLDOUT METRICS ===')
print(json.dumps(metrics, indent=2))

  ↑ s3://thesis-data-ismaktam/kriging/test/test_metrics.json
=== FINAL HOLDOUT METRICS ===
{
  "transform": "none",
  "variogram_model": "exponential",
  "tps_modes": [
    "2D",
    "3D"
  ],
  "per_mode": {
    "2D": {
      "all": {
        "crps": 0.39638986135055676,
        "mae": 0.49890324005729797,
        "rmse": 1.3416065608857461,
        "n": 11320920,
        "n_stations": 492
      },
      "predicted_wet": {
        "crps": 0.8876416385382193,
        "mae": 1.1400111347810755,
        "rmse": 2.061130674884886,
        "n": 4598899,
        "n_stations": 492
      }
    },
    "3D": {
      "all": {
        "crps": 0.38432061202167866,
        "mae": 0.47958311994959096,
        "rmse": 1.3105599906224183,
        "n": 11320920,
        "n_stations": 492
      },
      "predicted_wet": {
        "crps": 0.8579312633950017,
        "mae": 1.0924515899862017,
        "rmse": 2.011358995150492,
        "n": 4598899,
        "n_stations": 492
      }
    }
  },
  "kfold_me

## 7. S3 manifest

In [8]:
print('=== S3 ARTEFACTS (test outputs) ===')
for key in ['test/holdout_predictions.parquet', 'test/test_metrics.json', 'test/holdout_norms.pkl']:
    print(f'  s3://{S3_BUCKET}/{S3_ROOT}/{key}  '
          f'{"✓" if s3_exists(f"{S3_ROOT}/{key}") else "✗"}')

=== S3 ARTEFACTS (test outputs) ===
  s3://thesis-data-ismaktam/kriging/test/holdout_predictions.parquet  ✓
  s3://thesis-data-ismaktam/kriging/test/test_metrics.json  ✓
  s3://thesis-data-ismaktam/kriging/test/holdout_norms.pkl  ✓
